# IDM-VTON en Google Colab desde VS Code

Este cuaderno prepara la implementacion oficial de IDM-VTON para ejecutarla en un kernel remoto de Colab. No descarga paquetes, modelos ni imagenes hasta que ejecutes las celdas correspondientes.

## Conectar VS Code a Colab

1. Abre este archivo en VS Code e inicia sesion en la extension oficial **Google Colab**.
2. Selecciona **Select Kernel > Colab > Auto Connect** y autoriza tu cuenta Google.
3. En el runtime de Colab, selecciona una GPU si esta disponible. El nivel gratuito no garantiza GPU ni disponibilidad y puede desconectar sesiones.
4. Ejecuta las celdas en orden. La instalacion y los checkpoints pueden ocupar varios GB y tardar.

**Licencia:** el codigo y los checkpoints de IDM-VTON tienen licencia [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). No los uses comercialmente sin permiso.

**Privacidad:** el demo oficial inicia Gradio. Colab/Gradio puede mostrar un enlace publico temporal. No uses fotos sensibles ni compartas ese enlace; detén la app y desconecta el runtime al terminar.

Fuentes: [repositorio oficial](https://github.com/yisol/IDM-VTON), [modelo](https://huggingface.co/yisol/IDM-VTON), [demo y checkpoints de preprocesamiento](https://huggingface.co/spaces/yisol/IDM-VTON).

## 1. Comprobar el kernel remoto y la GPU

Esta celda debe ejecutarse en el kernel de Colab, no en el Python local del Mac. Si no hay CUDA, vuelve a la configuracion del runtime y solicita una GPU.

In [ ]:
import platform
import torch

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU CUDA en este kernel. En Colab selecciona una GPU en Entorno de ejecucion > Cambiar tipo de entorno de ejecucion y reconecta.")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA de PyTorch:", torch.version.cuda)

## 2. Descargar el codigo oficial

Se clona en el disco temporal de Colab. Al desconectar el runtime, este contenido puede borrarse.

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/IDM-VTON")
if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/yisol/IDM-VTON.git", str(repo_dir)],
        check=True,
    )
print("Repositorio:", repo_dir)

## 3. Instalar dependencias

El `environment.yaml` del proyecto especifica Python 3.10, PyTorch 2.0.1 y CUDA 11.8. Esta celda instala las dependencias Python de la demo, pero **no** reemplaza PyTorch ni modifica CUDA del runtime. Si aparecen incompatibilidades, revisa las versiones requeridas por el repositorio antes de cambiar el entorno.

In [ ]:
%pip install -q "numpy<2" accelerate==0.25.0 torchmetrics==1.2.1 tqdm==4.66.1 transformers==4.36.2 diffusers==0.25.0 einops==0.7.0 scipy==1.11.1 opencv-python gradio==4.24.0 fvcore cloudpickle omegaconf pycocotools basicsr av onnxruntime==1.16.2 huggingface_hub==0.20.2

## 4. Descargar checkpoints de preprocesamiento

Los cuatro archivos suman aproximadamente 1 GB. Se descargan desde el Space publico oficial de Hugging Face, sin guardar tokens ni montar Google Drive. Si Hugging Face solicita aceptar condiciones o iniciar sesion, hazlo desde tu propia cuenta; nunca pegues un token en una celda compartida.

In [ ]:
from huggingface_hub import hf_hub_download

checkpoint_files = [
    "ckpt/densepose/model_final_162be9.pkl",
    "ckpt/humanparsing/parsing_atr.onnx",
    "ckpt/humanparsing/parsing_lip.onnx",
    "ckpt/openpose/ckpts/body_pose_model.pth",
]
for filename in checkpoint_files:
    downloaded = hf_hub_download(
        repo_id="yisol/IDM-VTON",
        repo_type="space",
        filename=filename,
        local_dir=str(repo_dir),
    )
    size_mb = Path(downloaded).stat().st_size / (1024 * 1024)
    print(f"Listo: {filename} ({size_mb:.1f} MiB)")

## 5. Iniciar la demo

La demo oficial descarga el modelo IDM-VTON de Hugging Face durante el arranque, lo que puede requerir aceptar sus condiciones desde tu cuenta. La app puede mostrar un enlace Gradio temporal y publico; utiliza solo imagenes que tengas permiso para procesar y no sensibles.

In [ ]:
%cd /content/IDM-VTON
!python gradio_demo/app.py

Al terminar, detén la celda de Gradio y desconecta el runtime de Colab para liberar la sesión gratuita.